# Advanced Pandas: Combining Data
v.ekc

Real datasets rarely arrive as one table. Today we learn three ways to combine DataFrames: `merge`, `join`, and `concat` — and when each one is the right tool.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Merge |
| 3 | Join |
| 4 | Concatenate |
| 5 | 🔬 Activity — Humboldt Student Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

---
## 2. Merge

We'll use two DataFrames of Gilmore Girls characters as a toy example.



| Merge type | Code | What rows are kept |
|------------|------|--------------------|
| Inner (default) | `df1.merge(df2)` | Only rows matching in both |
| Outer | `df1.merge(df2, how='outer')` | All rows from both |
| Left | `df1.merge(df2, how='left')` | All rows from left df |
| Right | `df1.merge(df2, how='right')` | All rows from right df |
| Different col names | `df1.merge(df2, left_on='a', right_on='b')` | Specify each key column |
| Merge on index | `df1.merge(df2, left_on='col', right_index=True)` | Right df is indexed by key |
| Debug | `df1.merge(df2, how='outer', indicator=True)` | Adds `_merge` column showing origin |

In [ ]:
# Create the first dataframe
df1 = pd.DataFrame({
    'Name': ['Tyler', 'Dale', 'Kami', 'Emily'],
    'department': ['math', 'cs', 'math', 'math'],
    'office': ['BSS320B', 'BSS324', 'BSS356', 'BSS316']
})

# Create the second dataframe
df2 = pd.DataFrame({
    'Name': ['Tyler', 'Victoria', 'Kami', 'Chris'],
    'job': ['chair', 'admin coordinator', 'associate professor', 'professor'],
    'home': ['Arcata', 'McKinleyville', 'McKinleyville', 'Eureka'],
    'office': ['BSS320B', 'BSS320C', 'BSS356', 'BSS354']
})

In [ ]:
df1

In [ ]:
df2

In [ ]:
# A standard merge (inner)
df1.merge(df2)

In [ ]:
# Explicitly specifying what to merge by (same as before)
df1.merge(df2, on = 'Name')

In [ ]:
# What if they had different column names?
df2.rename(columns = {'Name':'naAaAme'},inplace=True)
df2

In [ ]:
df1.columns

In [ ]:
df2.columns

In [ ]:
# Try merging now (error!)
df1.merge(df2)


# Specify one of the column names (error!)
#    obviously this will not work either!!! no shared column name :(
# df1.merge(df2, on = 'Name')

In [ ]:
# How to specify both names if they are different
df1.merge(df2, left_on = 'Name',right_on = 'naAaAme')

In [ ]:
# Can drop the redundant column
df1.merge(df2, left_on = 'Name',right_on = 'naAaAme').drop(columns = 'naAaAme')

In [ ]:
# Reset it back to original
df2.rename(columns = {'naAaAme':'Name'},inplace=True)

In [ ]:
# Outer
df1.merge(df2, how = 'outer')

In [ ]:
# Use the indicator parameter to keep track of where the rows came from (good for debugging)
df1.merge(df2, how = 'outer',indicator=True)

Reset!

In [ ]:
df1

In [ ]:
df2

In [ ]:
# Left
df1.merge(df2,how = 'left')

In [ ]:
# Right
df1.merge(df2,how = 'right')

In [ ]:
# Cross join (less common, but occasionally handy)
df1.merge(df2,how='cross')

In [ ]:
# What if there are two common columns?
df1['School'] = ['UC Davis','Colorado State','UC Davis','UH']
df2['School'] = ['UC Davis','??','US Davis','UC Davis']

In [ ]:
df1

In [ ]:
df2

In [ ]:
# Test what a standard merge does
#   matches ALL columns

df1.merge(df2)

In [ ]:
# Outer merge behaves as expected
df1.merge(df2, how = 'outer')

In [ ]:
# What if we only specify one column to merge on?
df1.merge(df2, on = 'Name',how = 'outer')

In [ ]:
# Merge on both
df1.merge(df2, on = ['Name','office', 'School'],how = 'outer')

In [ ]:
# If one of the columns is actually an index
df2.set_index('Name',inplace=True)
df2

In [ ]:
# If one of the columns is actually an index
df1.merge(df2, left_on='Name', right_index=True)

In [ ]:
df2.reset_index(inplace=True)

---
## 3. Join

`.join()` is a shorthand for index-based merging — both DataFrames must be indexed by the key column.

In [ ]:
# Standard join won't work (error!)
df1.join(df2)

In [ ]:
# Join works when we just want to join on the index
df1.set_index('Name',inplace=True)
df2.set_index('Name',inplace=True)

In [ ]:
df1

In [ ]:
df2

In [ ]:
# still doesn't work (it doesn't know what to do with the identical columns) 
df1.join(df2)

In [ ]:
# We have to define suffixes when we use join (merge did it automatically)
df1.join(df2, lsuffix='_left',rsuffix='_right')

In [ ]:
# Can select different "how" with join
df1.join(df2,how='outer', lsuffix='_left',rsuffix='_right')

---
## 4. Concatenate

`pd.concat()` stacks DataFrames row-wise (or column-wise) — think of it as appending, not joining on a key.

### 📋 Board Reference

| What | Code | Notes |
|------|------|-------|
| Stack rows | `pd.concat([df1, df2])` | Default: `join='outer'` |
| Keep common cols only | `pd.concat([df1, df2], join='inner')` | |
| Reset index | `pd.concat([df1, df2], ignore_index=True)` | Avoids duplicate indices |
| Stack columns | `pd.concat([df1, df2], axis=1)` | Like cbind — be careful! |
| Add a row | `pd.concat([df, new_row], ignore_index=True)` | Better than deprecated `.append()` |

In [ ]:
# First lets reset the indices
df1.reset_index(inplace=True)
df2.reset_index(inplace=True)

In [ ]:
df1

In [ ]:
# A standard concatenate
pd.concat([df1,df2])

In [ ]:
# Explicitly state how to concatenate (keep all columns that appear in either dataset)
pd.concat([df1,df2],join='outer')

In [ ]:
# Only keep common columns
pd.concat([df1,df2],join='inner')

In [ ]:
# Left/right combinations are not supported
pd.concat([df1,df2],join='left')

In [ ]:
# Be careful with indices!
concat_gilmores = pd.concat([df1,df2])
concat_gilmores

In [ ]:
# indices might be duplicated
concat_gilmores.loc[2]

In [ ]:
# You can prevent this by ignoring the index (this will give new indices)
pd.concat([df1,df2],ignore_index=True)

In [ ]:
# concatenate the rows (can get very confusing if you aren't careful!)
pd.concat([df1,df2],axis=1)

In [ ]:
# Concat is useful for adding rows
new_row = pd.DataFrame([{'Name':'Kurk', 'Home':'Stars Hollow', 'School': 'Unknown'}])
new_row

In [ ]:
# Concat is useful for adding rows
df2 = pd.concat([df2,new_row],ignore_index=True)
df2

In [ ]:
# Concatenating columns would do an outer
pd.concat([df1,df2],axis=1)

In [ ]:
# Change it to inner
pd.concat([df1,df2],axis=1,join='inner')

Let's try reindexing by 'Name' since this is how `axis=1` aligns the dataframe!

In [ ]:
df1_idx = df1.set_index('Name')
df2_idx = df2.set_index('Name')

In [ ]:
df1_idx

In [ ]:
# this is default join='outer'
#    but now we don't get the double Rory and Lorelai because we are using their names

pd.concat([df1_idx, df2_idx], axis=1)

---
## 5. 🔬 Activity — Humboldt Student Data

Apply combining operations to real Cal Poly Humboldt institutional data.

---
### Try It 1: Merge Student Datasets 😊

1. **Load the data** — run the cells below to read in the Humboldt student datasets. Check what would happen if you did not include the `skiprows` argument.

2. **Merge** the two DataFrames. Try different `how` arguments — what changes?

3. **Recreate the figure** from the discussion question using the merged dataset.

In [ ]:
passing = pd.read_csv('humboldt_data/Humboldt_Passing_Fa23.csv',skiprows=5)

In [ ]:
passing.head()

In [ ]:
first_gen = pd.read_csv('humboldt_data/FirstGenData_Fa23.csv',skiprows=5)

In [ ]:
first_gen.head()

In [ ]:
# 2. Merge the two dataframes — try different how arguments

passing.merge(first_gen, left_on=['Course Code', 'Course Title'],right_on = ['Course Code', 'Course'], how='inner').drop(columns = 'Course')


In [ ]:
# 3. Recreate the figure from the discussion question


---
## Appendix — Combining Data Quick Reference

### Merge
```python
df1.merge(df2)                                          # inner (default)
df1.merge(df2, how='outer')                             # outer
df1.merge(df2, how='left')                              # left
df1.merge(df2, on='key_col')                            # explicit key
df1.merge(df2, left_on='a', right_on='b')               # different names
df1.merge(df2, left_on='col', right_index=True)         # right df indexed
df1.merge(df2, how='outer', indicator=True)             # debug: see source
```

### Join
```python
# Both dfs must be indexed by the key
df1.set_index('key').join(df2.set_index('key'),
                          lsuffix='_l', rsuffix='_r')
```

### Concat
```python
pd.concat([df1, df2])                       # stack rows (outer join)
pd.concat([df1, df2], join='inner')         # keep only common columns
pd.concat([df1, df2], ignore_index=True)    # reset index
pd.concat([df1, df2], axis=1)               # stack columns
pd.concat([df, new_row], ignore_index=True) # add a row
```

### Checklist flow?

```
Do I want to stack data?
→ YES → concat

NO → Do I want to match rows?

→ YES:
    Match by column? → merge
    Match by index? → join
    Match by time (approx)? → merge_asof```